# Dual-Heat / SLICE Continual Learning — Analysis Notebook

Visualizacao e comparacao de metodos de Continual Learning com LoRA.
Suporta resultados do **Qwen experiment** (4 tasks) e do **orchestrator completo**.

## Como usar

1. Execute `python -m cl_lora.qwen_experiment --compare-all` para gerar resultados
2. Ou aponte `RESULTS_DIR` para uma pasta com subpastas de metodos
3. Execute as celulas abaixo para visualizar

In [ ]:
import json, math, sys, os
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (10, 6)

print('Imports OK. Matplotlib backend:', matplotlib.get_backend())

## 1. Carregar resultados

Busca automaticamente por resultados salvos em `results/qwen_experiment/`.

In [ ]:
RESULTS_DIR = Path('results/qwen_experiment')

def load_qwen_results(base_dir: Path = RESULTS_DIR) -> Dict[str, Any]:
    if not base_dir.exists():
        print(f'Diretorio nao encontrado: {base_dir}')
        print('Execute primeiro: python -m cl_lora.qwen_experiment --compare-all')
        return {}

    results = {}
    for run_dir in sorted(base_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        metrics_file = run_dir / 'metrics.json'
        if not metrics_file.exists():
            continue
        with open(metrics_file) as f:
            data = json.load(f)

        label = data.get('method', run_dir.name)
        extra = data.get('cl_hyperparams', {})
        if extra.get('lateral_inhibition') is False:
            label = 'DualHeat (EWC only)'
        elif extra.get('slow_strength', 2.0) > 2.0:
            label = f'DualHeat (beta={extra["slow_strength"]})'

        results[label] = data
        ap = data['average_accuracy_ap']
        fp = data['final_performance_fp']
        fg = data['avg_forgetting']
        print(f'  Carregado: {label:35s} | AP={ap:.3f}  FP={fp:.3f}  Fgt={fg:.3f}')

    return results


results = load_qwen_results()

## 2. Heatmap da Matriz de Resultados

Cada celula [i, j] mostra a acuracia na task j depois de treinar ate a task i.
A diagonal principal mostra o AP. A ultima linha mostra o FP.

In [ ]:
def plot_results_heatmap(metrics: Dict[str, Any], title: str = None):
    matrix = metrics.get('results_matrix')
    task_order = [k for k in metrics.get('per_task_forgetting', {}).keys()]

    if not matrix or not task_order:
        if matrix:
            n = len(matrix)
            task_order = [f'Task {i+1}' for i in range(n)]
        else:
            print('Sem dados de matriz para plotar.')
            return None

    n_tasks = len(task_order)
    arr = np.array(matrix, dtype=float)

    fig, ax = plt.subplots(figsize=(n_tasks * 1.5 + 1, n_tasks * 1.2 + 1))
    mask = np.isnan(arr)
    cmap = sns.color_palette('viridis', as_cmap=True)

    sns.heatmap(
        arr, mask=mask, annot=True, fmt='.3f',
        cmap=cmap, vmin=0, vmax=1,
        xticklabels=task_order, yticklabels=task_order,
        ax=ax, cbar_kws={'label': 'Accuracy'},
        linewidths=0.5, linecolor='white',
    )

    ax.set_xlabel('Task Avaliada')
    ax.set_ylabel('Apos treinar ate')
    ax.set_title(title or f'{metrics.get("method", "?")} — Results Matrix',
                 fontsize=14, fontweight='bold', pad=12)

    plt.tight_layout()
    return fig


for label, data in results.items():
    fig = plot_results_heatmap(data, title=label)
    if fig:
        plt.show()
        print()

## 3. Comparacao de Metricas Agregadas (AP, FP, Forgetting)

In [ ]:
def plot_metric_comparison(results_dict: Dict[str, Any]):
    if not results_dict:
        print('Sem resultados para comparar.')
        return

    labels = list(results_dict.keys())
    ap_vals = [results_dict[k]['average_accuracy_ap'] for k in labels]
    fp_vals = [results_dict[k]['final_performance_fp'] for k in labels]
    fgt_vals = [results_dict[k]['avg_forgetting'] for k in labels]

    x = np.arange(len(labels))
    width = 0.25

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.8), 5.5))
    pal = sns.color_palette('muted')

    bars1 = ax.bar(x - width, ap_vals, width, label='AP (Avg Accuracy)', color=pal[0])
    bars2 = ax.bar(x, fp_vals, width, label='FP (Final Perf)', color=pal[2])
    bars3 = ax.bar(x + width, fgt_vals, width, label='Forgetting (down is better)', color=pal[3])

    for bars, vals in [(bars1, ap_vals), (bars2, fp_vals), (bars3, fgt_vals)]:
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Score')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha='right')
    ax.legend(loc='upper right', framealpha=0.9)
    ax.set_title('CL Method Comparison', fontsize=14, fontweight='bold')
    ymax = max(max(ap_vals), max(fp_vals), max(fgt_vals)) * 1.2 + 0.05
    ax.set_ylim(0, ymax)
    ax.axhline(y=0, color='grey', linewidth=0.5)

    plt.tight_layout()
    return fig


fig = plot_metric_comparison(results)
if fig:
    plt.show()

## 4. Forgetting por Task

Mostra o esquecimento (diag - final) para cada task individual.

In [ ]:
def plot_per_task_forgetting(results_dict: Dict[str, Any]):
    if not results_dict:
        return

    all_tasks = set()
    for data in results_dict.values():
        all_tasks.update(data.get('per_task_forgetting', {}).keys())
    all_tasks = sorted(all_tasks)
    if not all_tasks:
        print('Sem dados de forgetting por task.')
        return

    n_tasks = len(all_tasks)
    n_methods = len(results_dict)
    fig, ax = plt.subplots(figsize=(max(8, n_tasks * 1.5), 5))
    x = np.arange(n_tasks)
    width = min(0.8 / n_methods, 0.25)
    colors = sns.color_palette('muted', n_methods)

    for i, (label, data) in enumerate(results_dict.items()):
        forgetting = data.get('per_task_forgetting', {})
        vals = [forgetting.get(t, 0) for t in all_tasks]
        offset = (i - (n_methods - 1) / 2) * width
        bars = ax.bar(x + offset, vals, width, label=label, color=colors[i])
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

    ax.set_ylabel('Forgetting (diag - final)')
    ax.set_xlabel('Task')
    ax.set_xticks(x)
    ax.set_xticklabels(all_tasks, rotation=15)
    ax.legend(loc='best', framealpha=0.9, fontsize=8)
    ax.set_title('Forgetting per Task', fontsize=14, fontweight='bold')
    ax.axhline(y=0, color='grey', linewidth=0.5, linestyle='--')

    plt.tight_layout()
    return fig


fig = plot_per_task_forgetting(results)
if fig:
    plt.show()

## 5. Evolucao do Desempenho ao Longo das Tasks

Curvas mostrando como a acuracia em cada task evolui conforme novas tasks sao aprendidas.

In [ ]:
def plot_performance_evolution(results_dict: Dict[str, Any]):
    if not results_dict:
        return

    for label, data in results_dict.items():
        matrix = data.get('results_matrix')
        forgetting = data.get('per_task_forgetting', {})
        if not matrix or not forgetting:
            continue

        task_names = list(forgetting.keys())
        n_tasks = len(task_names)
        arr = np.array(matrix, dtype=float)

        fig, ax = plt.subplots(figsize=(9, 5))
        colors = sns.color_palette('husl', n_tasks)
        stages = list(range(1, n_tasks + 1))

        for i, task_name in enumerate(task_names):
            scores = arr[:, i]
            valid = ~np.isnan(scores)
            ax.plot(
                np.array(stages)[valid], scores[valid],
                marker='o', linewidth=2, markersize=6,
                label=task_name, color=colors[i],
            )
            if valid.any():
                last_idx = np.where(valid)[0][-1]
                ax.annotate(f'{scores[last_idx]:.3f}',
                            (stages[last_idx], scores[last_idx]),
                            textcoords='offset points', xytext=(5, 5),
                            fontsize=8, color=colors[i])

        ax.set_xlabel('Stage (after training through task N)')
        ax.set_ylabel('Accuracy')
        ax.set_title(f'Per-Task Evolution -- {label}', fontsize=13, fontweight='bold')
        ax.set_xticks(stages)
        ax.legend(loc='best', framealpha=0.9)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        print()

In [ ]:
plot_performance_evolution(results)

## 6. Radar Chart -- Perfil Multicriterio

Cada metodo visualizado como poligono em AP, FP, 1-Forgetting (e GP/IP se disponivel).

In [ ]:
def plot_radar_comparison(results_dict: Dict[str, Any]):
    if not results_dict:
        return

    dims = ['AP', 'FP', '1-Forgetting']

    # Check if GP/IP available
    has_gp = any(data.get('metrics', {}).get('gp') is not None
                 for data in results_dict.values())
    if has_gp:
        dims.extend(['GP', 'IP'])

    n = len(dims)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors = sns.color_palette('muted', len(results_dict))

    for i, (label, data) in enumerate(results_dict.items()):
        ap = data.get('average_accuracy_ap', 0)
        fp = data.get('final_performance_fp', 0)
        fgt = 1 - data.get('avg_forgetting', 0)
        vals = [ap, fp, fgt]
        if has_gp:
            m = data.get('metrics', {})
            vals.append(m.get('gp', 0))
            vals.append(m.get('ip', 0))
        vals += vals[:1]

        ax.fill(angles, vals, alpha=0.1, color=colors[i])
        ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dims, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_title('Multicriteria Method Profile', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), framealpha=0.9)

    plt.tight_layout()
    return fig


fig = plot_radar_comparison(results)
if fig:
    plt.show()

## 7. Tabela Comparativa

In [ ]:
def build_comparison_table(results_dict: Dict[str, Any]) -> pd.DataFrame:
    if not results_dict:
        return pd.DataFrame()

    rows = []
    for label, data in results_dict.items():
        row = {
            'Metodo': label,
            'AP (up)': data.get('average_accuracy_ap', float('nan')),
            'FP (up)': data.get('final_performance_fp', float('nan')),
            'Forgetting (down)': data.get('avg_forgetting', float('nan')),
        }
        for task, fgt in data.get('per_task_forgetting', {}).items():
            row[f'Fgt-{task}'] = fgt

        gp = data.get('metrics', {}).get('gp', None)
        ip = data.get('metrics', {}).get('ip', None)
        if gp is not None:
            row['GP (up)'] = gp
        if ip is not None:
            row['IP (up)'] = ip

        rows.append(row)

    df = pd.DataFrame(rows)
    if 'Metodo' in df.columns:
        df = df.set_index('Metodo')
    return df


df = build_comparison_table(results)
if not df.empty:
    print('\n=== COMPARISON TABLE ===')
    up_cols = [c for c in df.columns if '(up)' in c]
    fgt_cols = [c for c in df.columns if 'Fgt' in c or '(down)' in c]
    styled = df.style.format(precision=4)
    if up_cols:
        styled = styled.background_gradient(cmap='viridis', subset=up_cols, axis=0)
    if fgt_cols:
        styled = styled.background_gradient(cmap='RdYlGn_r', subset=fgt_cols, axis=0)
    display(styled)
else:
    print('Sem dados para tabela.')

## 8. Executar Experimentos do Notebook

Se nenhum resultado existir, esta celula executa o --compare-all.
Necessita de GPU com CUDA.

In [ ]:
def run_qwen_experiments(force: bool = False):
    if not force and RESULTS_DIR.exists() and any(RESULTS_DIR.iterdir()):
        print('Resultados ja existem. Use force=True para re-executar.')
        return

    print('Executando Qwen experiment --compare-all...')
    print('(30-60 min em GPU)')

    import subprocess, sys
    result = subprocess.run(
        [sys.executable, '-m', 'cl_lora.qwen_experiment', '--compare-all'],
        capture_output=True, text=True, cwd=Path.cwd()
    )
    out = result.stdout
    print(out[-2000:] if len(out) > 2000 else out)
    if result.returncode != 0:
        print(f'ERRO (codigo {result.returncode}):')
        err = result.stderr
        print(err[-1000:] if len(err) > 1000 else err)
    else:
        print('OK. Resultados em', RESULTS_DIR)


# Descomente para executar:
# run_qwen_experiments(force=False)

## 9. Salvar Todos os Graficos

Salva todos os graficos em `plots/`.

In [ ]:
def save_all_plots(results_dict: Dict[str, Any], output_dir: str = 'plots'):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    for label, data in results_dict.items():
        fig = plot_results_heatmap(data, title=label)
        if fig:
            safe = label.lower().replace(' ', '_').replace('(', '').replace(')', '')
            fig.savefig(out / f'heatmap_{safe}.png', bbox_inches='tight')
            plt.close(fig)

    fig = plot_metric_comparison(results_dict)
    if fig:
        fig.savefig(out / 'metric_comparison.png', bbox_inches='tight')
        plt.close(fig)

    fig = plot_per_task_forgetting(results_dict)
    if fig:
        fig.savefig(out / 'per_task_forgetting.png', bbox_inches='tight')
        plt.close(fig)

    fig = plot_radar_comparison(results_dict)
    if fig:
        fig.savefig(out / 'radar_comparison.png', bbox_inches='tight')
        plt.close(fig)

    df = build_comparison_table(results_dict)
    if not df.empty:
        df.to_csv(out / 'comparison_table.csv')

    print(f'Graficos salvos em {out.absolute()}/')
    for f in sorted(out.iterdir()):
        sz = f.stat().st_size
        print(f'  {f.name}  ({sz/1024:.1f} KB)')


save_all_plots(results)

## 10. Carregar Resultados do Orchestrator Completo

Aponte FULL_RESULTS_DIR para a pasta com as sequencias (ex: results/).

In [ ]:
def load_orchestrator_results(base_dir: Path) -> Dict[str, Any]:
    """Carrega resultados do orchestrator completo.

    Estrutura esperada:
        base_dir/<sequence>/<run>/metrics.json
    """
    if not base_dir.exists():
        print(f'Diretorio nao encontrado: {base_dir}')
        return {}

    results = {}
    for seq_dir in sorted(base_dir.iterdir()):
        if not seq_dir.is_dir():
            continue
        for run_dir in sorted(seq_dir.iterdir()):
            if not run_dir.is_dir():
                continue
            metrics_file = run_dir / 'metrics.json'
            if not metrics_file.exists():
                continue
            with open(metrics_file) as f:
                data = json.load(f)

            seq_name = seq_dir.name
            run_name = run_dir.name
            key = f'{seq_name}/{run_name}'
            results[key] = data
            ap = data.get('summary',{}).get('metrics',{}).get('ap','?')
            print(f'  {key:50s} | AP={ap}')

    return results


# FULL_RESULTS_DIR = Path('results')
# full_results = load_orchestrator_results(FULL_RESULTS_DIR)
# plot_metric_comparison(full_results)